In [2]:
import os
import numpy as np
import librosa
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import StandardScaler

In [3]:
# Path to dataset folder (containing subfolders per class)
DATASET_PATH = "/Users/ankitaupadhyaya/Documents/Audio Files/Organized Data/All Files"
SAMPLE_RATE = 22050
N_MFCC = 42

In [6]:
def extract_features(file_path):
    y, sr = librosa.load(file_path, sr=SAMPLE_RATE)
    
    # MFCC
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC)
    mfcc_mean = np.mean(mfcc.T, axis=0)

    # ZCR
    zcr = np.mean(librosa.feature.zero_crossing_rate(y).T)

    # Spectral centroid
    centroid = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr).T)

    # Combine
    features = np.concatenate([mfcc_mean, [zcr, centroid]])
    return features


In [8]:
# Prepare data
X, y = [], []
class_labels = sorted([
    f for f in os.listdir(DATASET_PATH) if os.path.isdir(os.path.join(DATASET_PATH, f))
])
label_map = {label: idx for idx, label in enumerate(class_labels)}

for label in class_labels:
    folder_path = os.path.join(DATASET_PATH, label)
    for file in os.listdir(folder_path):
        if file.endswith('.wav'):
            file_path = os.path.join(folder_path, file)
            try:
                feat = extract_features(file_path)
                X.append(feat)
                y.append(label_map[label])
            except Exception as e:
                print(f"Failed {file_path}: {e}")

X = np.array(X)
y = np.array(y)

In [9]:
# Scale
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [10]:
# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [11]:
# RF Model
model = RandomForestClassifier(
    n_estimators=150,
    max_depth=20,
    min_samples_leaf=3,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', max_depth=20,
                       min_samples_leaf=3, n_estimators=150, n_jobs=-1,
                       random_state=42)

In [16]:
# Eval
y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%")
print("Classification Report:\n", classification_report(y_test, y_pred, target_names=class_labels))


Accuracy: 49.17%
Classification Report:
               precision    recall  f1-score   support

           1       1.00      1.00      1.00         3
          10       0.67      0.50      0.57         4
         100       0.67      0.50      0.57         4
          11       1.00      0.50      0.67         4
          12       1.00      1.00      1.00         3
          13       0.50      1.00      0.67         3
          14       1.00      0.75      0.86         4
          15       0.60      0.75      0.67         4
          16       1.00      0.25      0.40         4
          17       0.60      1.00      0.75         3
          18       0.67      0.67      0.67         3
          19       0.40      0.67      0.50         3
           2       0.67      0.67      0.67         3
          20       0.50      0.50      0.50         4
          21       0.00      0.00      0.00         4
          22       0.00      0.00      0.00         3
          23       0.50      0.67      0

/Users/ankitaupadhyaya/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/ankitaupadhyaya/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/ankitaupadhyaya/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result)

In [20]:
import joblib

# Save the trained model
joblib.dump(model, 'NEP_RF_model.joblib')

# Save the scaler
joblib.dump(scaler, 'NEP_RF_scaler.joblib')


['NEP_RF_scaler.joblib']

In [28]:
import joblib
import librosa
import numpy as np
import os

# Load the trained model and scaler
model = joblib.load('NEP_RF_model.joblib')
scaler = joblib.load('NEP_RF_scaler.joblib')

# Label map (same as during training)
label_map = {0: '1', 1: '10', 2: '100', 3: '11', 4: '12', 5: '13', 6: '14', 7: '15', 8: '16', 9: '17', 10: '18',
             11: '19', 12: '2', 13: '20', 14: '21', 15: '22', 16: '23', 17: '24', 18: '25', 19: '26', 20: '27',
             21: '28', 22: '29', 23: '3', 24: '30', 25: '31', 26: '32', 27: '33', 28: '34', 29: '35', 30: '36',
             31: '37', 32: '38', 33: '39', 34: '4', 35: '40', 36: '41', 37: '42', 38: '43', 39: '44', 40: '45',
             41: '46', 42: '47', 43: '48', 44: '49', 45: '5', 46: '50', 47: '51', 48: '52', 49: '53', 50: '54',
             51: '55', 52: '56', 53: '57', 54: '58', 55: '59', 56: '6', 57: '60', 58: '61', 59: '62', 60: '63',
             61: '64', 62: '65', 63: '66', 64: '67', 65: '68', 66: '69', 67: '7', 68: '70', 69: '71', 70: '72',
             71: '73', 72: '74', 73: '75', 74: '76', 75: '77', 76: '78', 77: '79', 78: '8', 79: '80', 80: '81',
             81: '82', 82: '83', 83: '84', 84: '85', 85: '86', 86: '87', 87: '88', 88: '89', 89: '9', 90: '90',
             91: '91', 92: '92', 93: '93', 94: '94', 95: '95', 96: '96', 97: '97', 98: '98', 99: '99'}

# Feature extraction function (same as during training)
def extract_features(file_path):
    y, sr = librosa.load(file_path, sr=22050)
    
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=42)
    mfcc_mean = np.mean(mfcc.T, axis=0)

    zcr = np.mean(librosa.feature.zero_crossing_rate(y).T)
    centroid = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr).T)

    features = np.concatenate([mfcc_mean, [zcr, centroid]])
    return features.reshape(1, -1)

# Path to new .wav file
test_audio_path = '/Users/ankitaupadhyaya/Documents/Audio Files/Organized Data/All Files/1/NEP_S2_D(1)_r1.wav'  # Replace with your actual path

# Extract features
features = extract_features(test_audio_path)

# Scale features
scaled = scaler.transform(features)

# Predict
prediction = model.predict(scaled)[0]
predicted_label = label_map[prediction]

print(f"Predicted Digit: {predicted_label}")


Predicted Digit: 1
